# Jupyter Notebook 3 — Mutation Detection

**Purpose:** Trace loop traversals, detect list/dict/set mutation inside loop bodies, and write pattern matching scripts.

In [ ]:
import ast

unsafe_code = """
for item in my_list:
    if item == 'dirty':
        my_list.remove(item)
"""

tree = ast.parse(unsafe_code)

class LoopMutationDetector(ast.NodeVisitor):
    def visit_For(self, node):
        if isinstance(node.iter, ast.Name):
            iterable = node.iter.id
            for body_node in ast.walk(node):
                if isinstance(body_node, ast.Call) and isinstance(body_node.func, ast.Attribute):
                    if isinstance(body_node.func.value, ast.Name) and body_node.func.value.id == iterable:
                        if body_node.func.attr in ('remove', 'pop', 'append', 'clear'):
                            print(f"Flagged loop mutation on collection '{iterable}' at line {body_node.lineno}")

LoopMutationDetector().visit(tree)